# Lab 06 · Throttle tokens per caller with token limits

Publish a rate-limited alias that gives each caller 1,000 tokens per minute, then send a burst of requests, watch the
gateway answer `429` with `Retry-After`, and prove that a second key has its own allowance. This notebook is the
Python track of the [lab guide](README.md): it deploys the same [infra/limited-alias.bicep](infra/limited-alias.bicep) as the
Bicep + azd track and runs the same burst as [infra/trigger-token-limit.sh](infra/trigger-token-limit.sh).

The template doesn't touch the model or alias that your clients already call. It adds the alias
`gpt-6-luna-limited`, which targets the same backend model and carries its own token limit. Callers that you move
to it get the smaller allowance, and everyone else keeps calling `gpt-6-luna`.

**Check the model → Deploy the limited alias → Burst → Inspect 429 → Second key → Clean up**

## Architecture at a glance

| Component | What it does |
| --- | --- |
| Backend model `foundry-models/gpt-6-luna` | The model from notebook 02. Its own limit of 10,000 tokens per minute for each caller stays as it is. |
| Alias `gpt-6-luna-limited` | Targets the same model and carries a `tokenLimit` policy: 1,000 tokens per minute, counted per `identity`, so every runtime key gets its own counter. |
| Runtime key `app-first-call` | The caller that runs out of tokens. |
| Runtime key `lab06-second-key` | A second caller that still has a full allowance. |
| [infra/limited-alias.bicep](infra/limited-alias.bicep) | Declares the limited alias and its token limit. |
| [../shared/utils.py](../shared/utils.py) | Helpers that every lab notebook shares: settings, your `az login` sign-in, Bicep deployments, keys, and retries. |

![Token limits: each caller identity gets its own counter, and the gateway answers 429 when it runs out](../assets/diagrams/token-limits.svg)

## Prerequisites

- An Azure subscription where you can create resources (Contributor on the resource group) and assign roles
  (Owner, User Access Administrator, or Role Based Access Control Administrator).
- The [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli), signed in with `az login`. The notebook
  deploys Bicep with `az deployment group create`, and the CLI installs Bicep the first time it needs it.
- Python 3.10 or later with Jupyter, for example VS Code with the Python and Jupyter extensions.
- [01-provision-gateway.ipynb](../01-provision-gateway/01-provision-gateway.ipynb) and
  [02-first-model-call.ipynb](../02-first-model-call/02-first-model-call.ipynb) have run, so the gateway publishes
  the `gpt-6-luna` model.

Run the cells in order, or click **Run All**. Each step is idempotent: rerun any cell and it updates the same resources.

In [ ]:
%pip install -q --upgrade azure-identity requests openai

### 0. Initialize the lab

The defaults match notebooks 01 and 02, so `utils.Lab` finds the gateway without edits. The limited alias targets
the backend model `gpt-6-luna` under the `foundry-models` provider that notebook 02 published.

In [ ]:
import sys
sys.path.insert(0, "../shared")   # helpers that every lab notebook shares
import utils

lab = utils.Lab("06-token-limits")
provider        = "foundry-models"     # model provider from notebook 02
model           = "gpt-6-luna"         # backend model that the limited alias targets
second_key_name = "lab06-second-key"   # created and deleted by this notebook

### 1. Check the model and get a key

The cell confirms that notebook 02 published the model, then reuses the `app-first-call` key, or creates it.

In [ ]:
lab.require(f"{lab.workspace_id}/modelProviders/{provider}/models/{model}", "Run 02-first-model-call.ipynb first.")
print(f"✅ Model {provider}/{model} exists")

api_key, created_key = lab.api_key("app-first-call")
print(("✅ Created" if created_key else "♻️ Reusing") + f" key app-first-call: {api_key[:4]}…")

### 2. Deploy the limited alias

[infra/limited-alias.bicep](infra/limited-alias.bicep) creates the alias `<model>-limited` with a `policies` array that holds one
token limit. You set the allowance with the template's parameters:

- `allowedTokens` is how many tokens each caller gets per period.
- `period` accepts `minute`, `hour`, or `day`.
- `counterKey` is an array with `identity`, `ipAddress`, or both. `identity` is **Caller identity** in the portal:
  each runtime key or Microsoft Entra identity gets its own counter.

Limits stack: a request to the alias also counts against the model's own limit from notebook 02, and whichever runs
out first throttles the caller.

In [ ]:
import json

limited = lab.deploy("infra/limited-alias.bicep", gatewayName=lab.gateway_name, providerName=provider, modelName=model,
                     allowedTokens=1000, period="minute", counterKey=["identity"])
alias = limited["aliasName"]

policies = lab.arm("GET", f"{lab.workspace_id}/aliases/{alias}")["properties"]["policies"]
print(f"🔖 Alias {alias} policies:")
print(json.dumps(policies, indent=2))
limit = next(p for p in policies if p["type"] == "tokenLimit")
assert limit["count"] == 1000 and limit["period"] == "minute" and limit["counterKey"] == ["identity"]

### 3. Send a burst

Send eight requests to the limited alias that each use about 225 tokens. With a 1,000-token allowance,
the first few succeed and then the gateway answers `429` with `Retry-After`. The gateway counts tokens after each
response and its counters catch up within seconds, so a fast burst can overshoot the budget a little.

Successful responses also carry the Foundry deployment's own `x-ratelimit-remaining-tokens` header: the backend
still has capacity left while the gateway throttles this caller, which is the point of a per-caller limit.

The client sets `max_retries=0`: the OpenAI SDK normally honors `Retry-After` and retries for you, which would hide
the `429`. A new alias can take a minute to apply, so the cell first waits until the alias answers, and repeats the
burst if no request is throttled.

In [ ]:
import time
from openai import OpenAI, RateLimitError

PROMPT   = "Write a 150-word product update about an AI gateway that governs models and tools."
base_url = f"{lab.gateway_url}/default/models/openai/v1"
client   = OpenAI(base_url=base_url, api_key=api_key, max_retries=0)

def send(client, prompt=PROMPT, max_completion_tokens=250):
    """Send one chat request. Return the status, response headers, body text, and tokens used."""
    try:
        raw = client.chat.completions.with_raw_response.create(
            model=alias, max_completion_tokens=max_completion_tokens, reasoning_effort="none",
            messages=[{"role": "user", "content": prompt}])
        return raw.status_code, raw.headers, raw.text, raw.parse().usage.total_tokens
    except RateLimitError as error:
        return 429, error.response.headers, error.response.text, None

def burst(client, count=8):
    print(f"{'Request':<8} {'Status':<7} {'Retry-After':<12} {'Tokens used':<12} Backend tokens left")
    results = []
    for i in range(1, count + 1):
        status, headers, body, used = send(client)
        backend_left = headers.get("x-ratelimit-remaining-tokens", "-")
        print(f"{i:<8} {status:<7} {headers.get('retry-after', '-'):<12} {used or '-':<12} {backend_left}")
        results.append((status, headers, body))
    return results

utils.retry(lambda: client.chat.completions.create(   # wait until the new alias answers
    model=alias, max_completion_tokens=16, messages=[{"role": "user", "content": "ping"}]), statuses=(401, 404))

for attempt in range(4):
    results   = burst(client)
    throttled = [r for r in results if r[0] == 429]
    if throttled:
        break
    print("⏳ No 429 yet. The policy may still be applying; trying again in 20 s.")
    time.sleep(20)

assert throttled, "Expected at least one 429."
assert any(r[1].get("retry-after") for r in throttled), "A throttled response should carry Retry-After."
print(f"\n✅ {len(throttled)} of {len(results)} requests throttled with 429")

### 4. Inspect a throttled response

Look at the status, headers, and body of one throttled call. `Retry-After` tells the client how many seconds to wait.
Well-behaved SDKs, including the OpenAI SDKs, honor it automatically when they retry. The gateway rejected the
request itself: the backend didn't receive it.

In [ ]:
status, headers, body = throttled[-1]
print(f"📨 HTTP {status}")
for name, value in headers.items():
    print(f"   {name}: {value}")
print(f"   body: {body[:400]}")

### 5. Prove the counter is per caller

Create a second runtime key and call the limited alias while the first key is still throttled. The second key starts
with a full allowance, because **Caller identity** gives every key its own counter. A new key can take a moment to
work, so the cell retries `401` with a cheap request first.

In [ ]:
second_key, created_second = lab.api_key(second_key_name)
print(("✅ Created" if created_second else "♻️ Reusing") + f" key {second_key_name}: {second_key[:4]}…")
second_client = OpenAI(base_url=base_url, api_key=second_key, max_retries=0)
utils.retry(lambda: send(second_client, prompt="ping", max_completion_tokens=16), statuses=(401,), wait=15)

first_status = 200
for _ in range(6):   # use up the first key's allowance again
    first_status, *_ = send(client)
    if first_status == 429:
        break
second_status, *_ = send(second_client)

print(f"🔑 {'app-first-call':<18} HTTP {first_status}")
print(f"🔑 {second_key_name:<18} HTTP {second_status}")
assert first_status == 429 and second_status == 200
print("✅ Each key has its own token counter")

### 6. Clean up resources

The limited alias throttles anyone who calls it, so the cell deletes it, along with the second key. The model and its
`gpt-6-luna` alias were never changed. The cell deletes `app-first-call` only if this notebook created it.

To find throttled calls in **Monitoring**, configure Application Insights first (lab 13).

In [ ]:
lab.delete(f"{lab.workspace_id}/aliases/{alias}")
if created_second:
    lab.delete(f"{lab.gateway_id}/apiKeys/{second_key_name}")
if created_key:
    lab.delete(f"{lab.gateway_id}/apiKeys/app-first-call")

## Troubleshooting

- **No `429` after several bursts**: check that the request `model` is the limited alias, and that
  `lab.arm("GET", f"{lab.workspace_id}/aliases/{alias}")["properties"]["policies"]` lists the token limit.
- **`404` with `model_not_found` right after step 2**: a new alias takes up to a minute to apply. The burst cell waits
  for it; rerun the cell if it timed out.
- **`429` on the first request**: the key was throttled by an earlier run. Wait for `Retry-After` seconds.
- **`429` from the backend instead of the gateway**: the deployment's own quota ran out, and the body isn't
  `Token limit is exceeded`. Check `x-ratelimit-remaining-tokens` and `x-ratelimit-remaining-requests` on earlier
  responses, raise the deployment's capacity, or keep the sum of per-caller limits below it.
- **Every key is throttled together**: the counter isn't per caller. Deploy with `counterKey=["identity"]`.
- **The OpenAI SDK waits instead of failing**: it honors `Retry-After` and retries. Set `max_retries=0` to see the
  `429`.

For the other errors in this lab, see [Troubleshooting](../TROUBLESHOOTING.md#lab-06-token-limits).

## Next steps

- Cap spend in dollars per caller in [Enforce cost budgets with per-caller overrides](../07-cost-budgets/README.md).
- Go keyless in [Go keyless with Microsoft Entra ID and RBAC](../11-entra-rbac/README.md) and its notebook [11-entra-rbac.ipynb](../11-entra-rbac/11-entra-rbac.ipynb).
- Docs: [Policies in AI Gateway](https://ai.gateway.azure.com/docs/policies).
- Read this lab on the labs site: [https://azure.github.io/ai-gateway/labs/06-token-limits/](https://azure.github.io/ai-gateway/labs/06-token-limits/).